<a href="https://colab.research.google.com/github/vidaldominguez/uimp_deep_learning/blob/main/labs/P3_Localizaci%C3%B3n_de_objetos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Práctica 3: Localización de objetos**

La **localización de objetos** consiste identificar la ubicación de un objeto en una imagen y marcarlo con una caja delimitadora o *bounding box*. Es habitual que, además de localizar el objeto, se clasifique en una de las clases objetivo (clasificación y localización). Por su parte, el proceso de localización implica determinar las coordenadas del *bounding box*, por lo que se puede abordar como un problema de regresión.

A continuación, vamos a ver un ejemplo de clasificación y localización de objetos utilizando la librería [TensorFlow](https://www.tensorflow.org/). Para ello utilizaremos una red pre-entrenada en ImageNet como extractor de características y resolveremos un problema de aprendizaje multi-tarea: clasificación y regresión.

Antes de empezar, vamos a utilizar el método [set_random_seed()](https://www.tensorflow.org/api_docs/python/tf/keras/utils/set_random_seed) para establecer el valor de la **semilla** y garantizar la reproducibilidad de los resultados.

In [ ]:
# Fijar la semilla
from tensorflow.keras.utils import set_random_seed

seed = 121
set_random_seed(seed)  # establece todas las semillas aleatorias del programa (Python, NumPy y TensorFlow)

<hr>

### **1. Conjunto de datos**

En esta práctica vamos a utilizar un conjunto para clasificación y localización de objetos denominado  **CALTECH**, que es en realidad un subconjunto del original y que está disponible para descarga en [Zenodo](https://zenodo.org/records/4126613). Para ello, es necesario descargar el fichero [`CALTECH.zip`](https://drive.google.com/file/d/1NsJZ82l8wvG6_nX9EPTCjF7YQcBsjxIT/view?usp=sharing) que contiene:

*   Una carpeta `CALTECH_Dataset` con tres directorios, uno para cada clase:
> *  `elephant`, con 64 imágenes.
> *  `cougar_face`, con 66 imágenes.
> *  `butterfly`, con 88 imágenes.
*   Una carpeta `CALTECH_Annotation` con tres directorios, uno para cada clase, en los que hay un fichero de anotaciones para cada imagen. El fichero de anotaciones contiene las coordenadas con la localización del objeto en la imagen correspondiente.

In [ ]:
from google.colab import drive

# Montar el Google Drive en el directorio del proyecto y descomprimir el fichero con los datos
drive.mount('/content/gdrive')
!unzip -n '/content/gdrive/My Drive/datasets/CALTECH.zip' >> /dev/null  # ACTUALIZAR: ruta al fichero comprimido

#### **1.1. Anotaciones**

Las anotaciones del conjunto de datos están en formato `MAT`, un formato de archivo matricial utilizado en MATLAB.

Teniendo en cuenta esto definiremos, en primer lugar, una función que extrae la etiqueta de clase y las coordenadas del bounding box `(x1, y1, x2, y2)` de un archivo MAT, entre otros datos de interés.

In [ ]:
import scipy
import cv2

def extract_mat_contents(annot_directory, image_dir):
  mat = scipy.io.loadmat(annot_directory)

  # Obtener dimensiones de la imagen y coordenadas del bounding box
  height, width = cv2.imread(image_dir).shape[:2]
  x1, y2, y1, x2 = tuple(map(tuple, mat['box_coord']))[0]

  # Obtener la clase, que se indica en el nombre del directorio
  class_name = image_dir.split('/')[2]
  filename = '/'.join(image_dir.split('/')[-2:])

  return filename, width, height, class_name, x1, y1, x2, y2

A continuación, vamos a definir otra función que nos permita extraer toda la información y almacenarla en una variable de tipo `DataFrame`.

In [ ]:
import os
import pandas as pd

def mat_to_csv(annot_directory, image_directory, classes_folders):

  mat_list = []

  # Recorrer todas las carpetas, una para cada clase objetivo
  for class_folder in classes_folders:
    image_dir = os.path.join(image_directory, class_folder)
    annot_dir = os.path.join(annot_directory, class_folder)

    mat_files = sorted(os.listdir(annot_dir))
    img_files = sorted(os.listdir(image_dir))

    # Procesar todos los archivos MAT y sus correspondientes imágenes
    for mat, image_file in zip(mat_files, img_files):
      mat_path = os.path.join(annot_dir, mat)
      img_path = os.path.join(image_dir, image_file)
      value = extract_mat_contents(mat_path, img_path)
      mat_list.append(value)

  # Almacenar la información extraída en un DataFrame
  column_name = ['filename', 'width', 'height', 'class', 'xmin', 'ymin', 'xmax', 'ymax']
  df_mat = pd.DataFrame(mat_list, columns=column_name)

  return df_mat

Por último, utilizaremos estas dos funciones almacenar toda la información en un `DataFrame`, que convertiremos también a un fichero CSV. En este punto, cabe mencionar que si guardarmos dicho fichero en Drive, podremos ahorrarnos el este primer paso en próximas ejecuciones.

In [ ]:
# Especificar las rutas al directorio con las imágenes y las anotaciones
image_directory = 'CALTECH/CALTECH_Dataset'
annot_directory = 'CALTECH/CALTECH_Annotations'
classes_list = sorted(['butterfly',  'cougar_face', 'elephant'])

# Crear el DataFrame y el fichero CSV
labels_df = mat_to_csv(annot_directory, image_directory, classes_list)
labels_df.to_csv(('labels.csv'), index=None)

#### **1.2. Imágenes**

Una vez extraída toda la información de las anotaciones, el siguiente paso consiste en procesar las imágenes.

Para ello definiremos, en primer lugar, una función que lea las imágenes y pre-procese los datos, que incluye:

*   Normalizar las coordenadas de los *bounding boxes*.
*   Redimensionar y normalizar las imágenes.

In [ ]:
def preprocess_dataset(image_folder, classes_list, df, img_width, img_height):

  labels = []
  boxes = []
  img_list = []

  # Obtener las dimensiones y las etiquetas
  h = df['height']
  w = df['width']
  labels = list(df['class'])

  # Obtener las coordenadas de los bounding boxes y normalizarlos
  for x1, y1, x2, y2 in zip(list(df['xmin']/w), list(df['ymin']/h),list(df['xmax']/w), list(df['ymax']/h)):
    arr = [x1, y1, x2, y2]
    boxes.append(arr)

  # Recorrer todas las carpetas, una para cada clase objetivo
  for class_folder in classes_list:
    image_dir = os.path.join(image_folder, class_folder)
    img_files = sorted(os.listdir(image_dir))

    # Procesar todas las imágenes imágenes
    for image_file in img_files:
      img_path = os.path.join(image_dir, image_file)
      img  = cv2.imread(img_path)
      image = cv2.resize(img, (img_width, img_height))   # tamaño de imagen fijo
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)     # formato RGB
      image = image.astype("float") / 255.0              # normalización
      img_list.append(image)

  return labels, boxes, img_list

Con la función previamente definida podemos almacenar, en tres variables diferentes, las imágenes, sus etiquetas y las coordenadas de sus *bounding boxes*.

Y por último, codificaremos las clases objetivo mediante la técnica *one-hot*.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

# Especificar información dependiente del conjunto de datos
img_width = img_height = 224  # dimensiones de la imagen
n_channels = 3                # número de canales (RGB)
n_classes = 3                 # número de clases

labels, boxes, img_list = preprocess_dataset('CALTECH/CALTECH_Dataset', classes_list, labels_df, img_width, img_height)

# Codificar las etiquetas (one-hot)
label_encoder = LabelEncoder()
integer_labels = label_encoder.fit_transform(labels)
onehot_labels = tf.keras.utils.to_categorical(integer_labels)

#### **1.3. División en entrenamiento y validación**

Una vez obtenida toda la información del conjunto de datos, vamos a realizar la división en las particiones de **entrenamiento y validaciónt** (proporción 90:10). Para ello utilizaremos el método [train_test_split()](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), disponible en la librería `scikit-learn`.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import random

# Barajar los datos
combined_list = list(zip(img_list, boxes, onehot_labels))
random.shuffle(combined_list)
img_list, boxes, onehot_labels = zip(*combined_list)

# Dividir el conjunto de datos (imágenes, etiquetas, coordenadas de los bounding boxes)
train_images, val_images, train_labels, val_labels, train_boxes, val_boxes = train_test_split(np.array(img_list),
                                                                                              np.array(onehot_labels),
                                                                                              np.array(boxes),
                                                                                              test_size = 0.1,
                                                                                              random_state = seed)

print(f'Número de ejemplos del conjunto de entrenamiento: {len(train_images)}')
print(f'Número de ejemplos del conjunto de validación: {len(val_images)}')

<hr>

### **2. Red convolucional**

La CNN que vamos a definir está compuesta por la base convolucional de una [VGG16](https://www.tensorflow.org/api_docs/python/tf/keras/applications/vgg16), pre-entrenada con [ImageNet](https://www.image-net.org/). Los parámetros de esta parte del modelo permanecerán fijos, por lo que utilizaremos la VGG16 pre-entrenada como extractor de características.

Esa base convolucional irá seguida de una capa de [GlobalAveragePooling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling2D) y de dos ramas, para resolver cada una de las tareas:

*   Clasificación. Esta rama estará formada por cuatro capas completamente conectadas con la función de activación `relu` (tamaños: 256, 128, 64, 32), seguidas de una capa de salida que resuelva el problema de clasificación multi-clase (3 clases).
*   Regresión. Esta rama estará formada por cuatro capas completamente conectadas con la función de activación `relu` (tamaños: 256, 128, 64, 32), seguidas de una capa de salida que resuelva el problema de regresión (4 coordenadas).

A continuación, vamos a definir una función `get_model()` que cree el modelo con los elementos previamente descritos.

In [ ]:
# To-Do: Crear el modelo (base convolucional VGG16 + GAP + dos ramas: clasificación y regresión)

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

def get_model():

  # Cargar la base convolucional del modelo VGG16 pre-entrenado en ImageNet
  base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_width,img_height,3))

  # Ajustar los parámetros de las nuevas capas del modelo, dejando fijos los parámetros del resto de capas
  for layer in base_model.layers:
      layer.trainable = False   # por defecto, layer.trainable es True

  # Añadir nuevas capas a continuación de la base convolucional, para resolver la tarea de aprendizaje
  x = base_model.output
  x = GlobalAveragePooling2D()(x)

  # Crear la rama de clasificación (3 clases)
  x_class = Dense(256, activation="relu")(x)
  x_class = Dense(128, activation="relu")(x_class)
  x_class = Dense(64, activation="relu")(x_class)
  x_class  = Dense(32, activation="relu")(x_class)
  x_class = Dense(n_classes, activation='softmax',name="class_output")(x_class)

  # Crear la rama de regresión (4 coordenadas)
  x_box = Dense(256, activation="relu")(x)
  x_box = Dense(128, activation="relu")(x_box)
  x_box = Dense(64, activation="relu")(x_box)
  x_box = Dense(32, activation="relu")(x_box)
  x_box = Dense(4, activation='sigmoid', name= "box_output")(x_box)

  # Crear el modelo final e imprimir su representacion en modo texto
  model = Model(inputs=base_model.input, outputs=[x_class, x_box])

  return model

Por último, vamos a crear el modelo invocando la función anterior y a utilizar el método [`summary()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#summary) para imprimir una representación en modo texto de la arquitectura definida. Con este método es posible visualizar también el número de parámetros de cada capa de la red.

In [ ]:
# Crear el modelo e imprimir su representación en modo texto
model = get_model()
model.summary()

<hr>

### **3. Entrenamiento**

Una vez definida la arquitectura de la CNN, el siguiente paso es configurar el modelo para el entrenamiento. Para ello utilizaremos el método [`compile()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile), siendo estos algunos de sus parámetros más relevantes:

* `optimizer`: nombre del optimizador (`Adam`, `RMSProp`, etc.) y tasa de aprendizaje (`learning_rate`). En la web de `TensorFlow` puedes encontrar otros [optimizadores](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers).
* `loss`: función de pérdida (`mean_squared_error`, `binary_crossentropy`, `categorical_crossentropy`, etc.). En la web de `TensorFlow` puedes encontrar otras [funciones de pérdida](https://www.tensorflow.org/api_docs/python/tf/keras/losses).
* `loss_weights`: si el modelo tiene varias salidas, es posible ponderar las contribuciones de las diferentes funciones de pérdida utilizadas.
* `metrics`: métricas que se evalúan para los datos de entrenamiento y validación (`accuracy`, etc.). En la web de `TensorFlow` puedes encontrar otras [métricas](https://www.tensorflow.org/api_docs/python/tf/keras/metrics).

Para configurar el proceso de aprendizaje, utiliza el optimizador [SGD](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD) y una tasa de aprendizaje de `0.0001`. Con respecto a la función de pérdida y a la métrica de evaluación, se deberán seleccionar teniendo en cuenta la tarea a resolver.

In [ ]:
# Configurar el proceso de aprendizaje
from tensorflow.keras.optimizers import SGD

# Aprendizaje multi-tarea: clasificación + regresión
losses = {'class_output': 'categorical_crossentropy', 'box_output': 'mean_squared_error'}
loss_weights = {'class_output': 1.0, 'box_output': 1.0}
metrics = {'class_output': 'accuracy', 'box_output':  'mse'}

# Compilar el modelo con el optimizador SGD
opt = SGD(learning_rate = 1e-3, momentum = 0.9)
model.compile(optimizer = opt, loss = losses, loss_weights = loss_weights, metrics = metrics)

En algunos problemas puede ser interesante reducir la tasa de aprendizaje durante el entrenamiento, cuando una métrica haya dejado de mejorar. Para ello es posible definir el callback de [ReduceLROnPlateau](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau), con los siguientes parámetros:

* `monitor`: métrica que se monitoriza.
* `factor`: factor por el que se reduce la tasa de aprendizaje.
* `patience`: número de épocas sin mejora tras las que se reduce la tasa de aprendizaje.
* `min_lr`: límite inferior de la tasa de aprendizaje.

A continuación, definiremos este callback para reducir la tasa de aprendizaje en un factor de 0.0002 si la pérdida de validación no mejora durante 10 épocas consecutivas. El valor mínimo permitido para la tasa de aprendizaje será de `1e-7`.

In [ ]:
# Modificar el ritmo de aprendizaje en función de la pérdida de validación
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor = "val_loss", factor = 0.0002, patience = 10, min_lr = 1e-7, verbose = 1)

A continuación, vamos a entrenar el modelo para buscar los parámetros que hagan mínima la función de pérdida. Para ello utilizaremos el método [`fit()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit), que necesita que le suministremos los datos de entrenamiento y validación, y el número de *epochs*.

In [ ]:
# Entrenar el modelo
history = model.fit(x = train_images,
                    y= {"class_output": train_labels, "box_output": train_boxes},
                    validation_data=(val_images, {"class_output": val_labels, "box_output": val_boxes}),
                    batch_size = 32,
                    epochs = 30,
                    callbacks=[reduce_lr])

Por último, vamos a imprimir el error mínimo de entrenamiento y validación accediendo al atributo `history`.

In [ ]:
# Imprimir el error mínimo (entrenamiento y validación)
train_trace = np.array(history.history['loss'])
print(f'\nError mínimo en entrenamiento: {min(train_trace):.6f}')

val_trace = np.array(history.history['val_loss'])
print(f'Error mínimo en validación: {min(val_trace):.6f}')

Si el modelo entrenado funciona correctamente, es posible guardarlo con el método [`save()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#save) y cargarlo posteriormente con el método [`load_model()`](https://www.tensorflow.org/api_docs/python/tf/keras/models/load_model). De esta forma, evitamos tener que entrenarlo de nuevo con el ahorro computacional que conlleva.

In [ ]:
# Guardar el modelo completo como un fichero comprimido con extensión .keras
model.save('caltech.keras')

# Cargar el modelo previamente guardado
# model = tf.keras.models.load_model('caltech.keras')

<hr>

### **4. Predicción**

Una vez entrenado el modelo, es posible utilizarlo para clasificar y localizar objetos en nuevas imágenes.

Para ello, vamos a definir una función que pre-procese la nueva imagen con el mismo procedimiento que el aplicado a las imágenes de entrenamiento y validación.

In [ ]:
def preprocess(img, img_width, img_height):
    image = cv2.resize(img, (img_width, img_height))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float") / 255.0
    image = np.expand_dims(image, axis=0)

    return image

De manera equivalente, definiremos una función que post-procese las predicciones del modelo y transforme las coordenadas del *bounding box* a escala real para poder visualizar los resultados.

In [ ]:
def postprocess(image, results):
    class_probs, bounding_box = results
    class_index = np.argmax(class_probs)
    class_label = sorted(classes_list)[class_index]  # lista usada para las anotaciones

    # Extraer las coordenadas y transformarlas a su escala real
    h, w = image.shape[:2]
    x1, y1, x2, y2 = bounding_box[0]
    x1 = int(w * x1)
    x2 = int(w * x2)
    y1 = int(h * y1)
    y2 = int(h * y2)

    return class_label, class_probs, (x1, y1, x2, y2)

A continuación, vamos a definir la función que, dada una imagen, aplica el modelo para obtener las predicciones (probabilidades de clase y coordenadas del *bounding box*) y las visualiza.

In [ ]:
import matplotlib.pyplot as plt

def predict(image, img_width, img_height, returnimage = False,  scale = 0.8):

  # Pre-procesar la imagen y obtener la predicción
  processed_image = preprocess(image, img_width, img_height)
  results = model.predict(processed_image)

  # Post-procesar los resultados y anotar la imagen
  label, confidence, (x1, y1, x2, y2) = postprocess(image, results)
  cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 100), 2)
  cv2.putText(image, '{}'.format(label, confidence), (x1, y2 + int(35 * scale)), cv2.FONT_HERSHEY_COMPLEX, scale, (200, 55, 100), 2)

  plt.figure(figsize=(10,10))
  plt.imshow(image[:,:,::-1])

Por último, descargaremos una [imagen de Internet](https://cdn.pixabay.com/photo/2019/08/10/08/44/butterfly-4396444_1280.jpg), no vista durante el entrenamiento, para mostrar un ejemplo de predicción con el modelo previamente entrenado.

In [ ]:
!wget -q -O  butterfly.jpg https://cdn.pixabay.com/photo/2019/08/10/08/44/butterfly-4396444_1280.jpg
image = cv2.imread('/content/butterfly.jpg' )
predict(image, img_width, img_height, scale = 1)

### **5. Ejercicios**

A continuación, se propone un ejercicio para su resolución.

**EJERCICIO 1**

En el modelo que hemos definido, las dos ramas (clasificación y regresión) son totalmente independientes, mientras que la red VGG16 funciona de extractor de características (sus parámetros no se modifican durante el entrenamiento). Dicho de otra forma, las dos tareas no comparten parámetros "entrenables" pese a que sus funciones de pérdida se minimizan en un mismo proceso de entrenamiento.

Modifica el modelo original de tal manera que la bifurcación de las dos ramas se produzca justo antes de las capas de salida. ¿Qué implicaciones tiene este cambio en términos de complejidad del modelo y del proceso de aprendizaje?

In [ ]:
# To-Do: Solución al ejercicio 1